# Session 13 Solutions: Grouping and Combining Tables

Data Science with Python, Level 1, Stage 2: Working with Data

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
df["revenue"] = df["unit_price"] * df["quantity"]
df["date"] = pd.to_datetime(df["date"])

customers = pd.read_csv(DATA_URL + "shop_customers.csv")
products = pd.read_csv(DATA_URL + "shop_products.csv")
orders = pd.read_csv(DATA_URL + "shop_orders.csv")

## In-Class Exercises

### In-Class Exercise 1: Revenue per Item

In [2]:
df.groupby("item")["revenue"].sum().sort_values(ascending=False).head(5)

item
Chicken Sandwich    1014.0
Latte                885.5
Cappuccino           700.0
Veggie Sandwich      478.5
Espresso             477.5
Name: revenue, dtype: float64

### In-Class Exercise 2: Branch Summary

In [3]:
df.groupby("branch").agg(orders=("order_id", "count"), avg_qty=("quantity", "mean"),
                         items=("item", "nunique"), revenue=("revenue", "sum")).round(2)

,orders,avg_qty,items,revenue
branch,,,,
Campus,355,1.69,12,2041.5
Downtown,410,1.66,12,2374.2
Mall,235,1.61,12,1307.4


### In-Class Exercise 3: Product Names on Orders

In [4]:
lines = orders.merge(products, on="product_id")
lines.groupby("product")["quantity"].sum().sort_values(ascending=False).head(5)

product
Index Cards               89
Highlighters              86
Sticky Notes              84
Pen Set                   76
Statistics Made Simple    73
Name: quantity, dtype: int64

### In-Class Exercise 4: Check the Join

In [5]:
check = orders.merge(customers, on="customer_id", how="left", indicator=True)
lost = check[check["_merge"] == "left_only"]
print(len(lost), lost["order_id"].tolist())

5 [70056, 70244, 70455, 70739, 70877]


## Part A: Basic

### Exercise 1: Average GPA per Major

In [6]:
students = pd.read_csv(DATA_URL + "students.csv")
students.groupby("major").agg(avg_gpa=("gpa", "mean"), students=("name", "count")).round(2).sort_values("avg_gpa", ascending=False)

,avg_gpa,students
major,,
Design,3.16,5
Business,3.08,7
Engineering,3.06,5
Computer Science,3.01,7
Medicine,2.67,6


### Exercise 2: Customer Names on Orders

In [7]:
m = orders.merge(customers, on="customer_id", how="left")
print(m[["order_id", "name", "city", "quantity"]].head())
print(m["city"].value_counts(dropna=False))

   order_id            name    city  quantity
0     70001  Khaled Darwish    Homs         4
1     70002       Leen Issa  Aleppo         1
2     70003      Firas Issa    Homs         3
3     70004       Lea Karam    Homs         4
4     70005     Tala Khoury  Aleppo         2
city
Damascus    304
Aleppo      269
Homs        139
Latakia      93
Tartus       90
NaN           5
Name: count, dtype: int64


## Part B: Intermediate

### Exercise 3: Weather by Month

In [8]:
w = pd.read_csv(DATA_URL + "weather_2024.csv")
w["date"] = pd.to_datetime(w["date"])
w["month"] = w["date"].dt.month
temp = w.groupby(["month", "city"])["temp_max"].mean().unstack().round(1)
rain = w.groupby(["month", "city"])["rain_mm"].sum().unstack().round(0)
print(temp)
print(rain.idxmax())

city   Mountain Town  Port City
month                          
1                7.6       17.5
2                9.2       17.7
3               11.8       20.7
4               16.1       23.7
5               22.0       26.8
6               25.6       30.0
7               27.2       30.4
8               25.5       29.3
9               21.5       27.4
10              16.9       23.7
11              11.9       21.0
12               8.3       18.9
city
Mountain Town    1
Port City        2
dtype: int32


### Exercise 4: Best Customers

In [9]:
full = orders.merge(products, on="product_id").merge(customers, on="customer_id")
full["total"] = full["price"] * full["quantity"]
best = full.groupby(["customer_id", "name", "city", "segment"]).agg(
    orders=("order_id", "count"), spent=("total", "sum")).round(2)
best.sort_values("spent", ascending=False).head(10)

,,,,orders,spent
customer_id,name,city,segment,,
C276,Khaled Halabi,Aleppo,Regular,7,2310.92
C220,Nour Najjar,Damascus,Regular,8,1967.93
C317,Hala Zein,Aleppo,Regular,8,1758.36
C314,Ziad Issa,Tartus,Regular,7,1490.66
C329,Rana Karam,Aleppo,Regular,10,1223.48
C243,Fadi Nassar,Damascus,Premium,3,1172.86
C294,Nour Karam,Damascus,Regular,6,1096.34
C232,Sami Aziz,Tartus,Regular,11,1043.90
C334,Yara Zein,Tartus,Regular,12,1016.10


## Part C: Challenge

### Exercise 5: Above the Department Average

`assign` adds a column to a copy of the table, which is handy inside a chain of methods.

In [10]:
emp = pd.read_csv(DATA_URL + "employees.csv")
emp["vs_group"] = emp["monthly_salary"] - emp.groupby(["department", "level"])["monthly_salary"].transform("mean")
print(emp.sort_values("vs_group", ascending=False).head(5)[["name", "department", "level", "vs_group"]].round(0))
print(emp.assign(above=emp["vs_group"] > 0).groupby("city")["above"].mean().round(2))

              name   department   level  vs_group
216   Ruba Sabbagh  Engineering  Senior    1341.0
235   Sofia Nassar         Data     Mid    1263.0
121  Basel Mansour         Data  Senior    1252.0
179    Fadi Kassem  Engineering     Mid    1186.0
372    Noah Kassem  Engineering     Mid    1146.0
city
Amman     0.29
Beirut    0.11
Cairo     0.02
Dubai     0.98
Name: above, dtype: float64


## Part D: Cumulative Project: Cafe Sales Analysis

### Exercise 6: Monthly Cafe Report

In [11]:
df["month"] = df["date"].dt.month
print(df.groupby("month").agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
                              avg_order=("revenue", "mean")).round(2))
print(pd.pivot_table(df, index="month", columns="branch", values="revenue", aggfunc="sum", margins=True).round(1))
by_item = df.groupby(["month", "item"])["revenue"].sum().sort_values(ascending=False)
print(by_item.groupby(level=0).head(1))
print(pd.crosstab(df["branch"], df["payment"], normalize="index").round(2))

       orders  revenue  avg_order
month                            
1         348   1999.2       5.74
2         342   1903.6       5.57
3         310   1820.3       5.87
branch  Campus  Downtown    Mall     All
month                                   
1        714.5     827.9   456.8  1999.2
2        683.1     773.1   447.4  1903.6
3        643.9     773.2   403.2  1820.3
All     2041.5    2374.2  1307.4  5723.1
month  item            
1      Chicken Sandwich    416.0
2      Chicken Sandwich    331.5
3      Latte               280.0
Name: revenue, dtype: float64
payment   Card  Cash  Mobile
branch                      
Campus    0.57  0.29    0.13
Downtown  0.53  0.31    0.16
Mall      0.60  0.31    0.10


### Exercise 7: The Cafe Menu Table

In [12]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
menu = pd.DataFrame({
    "item": ["Espresso", "Latte", "Cappuccino", "Mocha", "Green Tea", "Black Tea", "Croissant",
             "Muffin", "Cookie", "Chicken Sandwich", "Veggie Sandwich", "Orange Juice"],
    "cost": [0.6, 0.9, 0.9, 1.1, 0.5, 0.4, 0.9, 1.0, 0.4, 2.6, 2.1, 1.4],
})
m = df.merge(menu, on="item", how="left", indicator=True)
assert (m["_merge"] == "both").all()
m["profit"] = (m["unit_price"] - m["cost"]) * m["quantity"]
print("Total profit:", round(m["profit"].sum(), 2))
print(m.groupby("branch")["profit"].sum().round(2))
print(m.groupby("item")["profit"].sum().sort_values(ascending=False).head(3).round(2))
# Coffee drinks bring most of the profit, especially lattes and cappuccinos.
# The Mall branch earns the least, so a coffee promotion there could pay off.

Total profit: 3974.4
branch
Campus      1429.8
Downtown    1639.6
Mall         905.0
Name: profit, dtype: float64
item
Latte               657.8
Chicken Sandwich    608.4
Cappuccino          520.0
Name: profit, dtype: float64


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | It splits by city and averages the sales in each group. |
| 2 | B | The inner level becomes the column headers. |
| 3 | B | The default join is inner: only matching keys are kept. |
| 4 | B | A left join keeps every row of the left table. |
| 5 | B | Each repeated key produces one row per match. |